# Profit evidence

Inspect the unified append-only bet ledger read-only. Keep paper and real evidence, recommendation and exploration lanes, and currencies separate. Probability quality remains the first gate; fixture-clustered ROI, CLV, and drawdown show whether an apparent edge survived quoted prices.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
DB = ROOT / 'data/state/oracle_bets.db'
connection = sqlite3.connect(f'file:{DB}?mode=ro', uri=True)
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", connection)
display(tables)

In [ ]:
bets = pd.read_sql_query('SELECT * FROM bets', connection)
events = pd.read_sql_query('SELECT * FROM bet_events ORDER BY event_at', connection)
print(f"Bets: {len(bets):,} | lifecycle events: {len(events):,}")
display(bets.tail(20))
display(events.tail(20))

In [ ]:
settlements = events.loc[events['event_type'].eq('settlement')].copy()
if settlements.empty:
    print('No settled bets yet; do not interpret theoretical edges as profit evidence.')
else:
    payload = settlements['payload_json'].map(__import__('json').loads).apply(pd.Series)
    settled = settlements[['bet_id', 'event_at']].join(payload)
    settled = settled.merge(bets[['id', 'mode', 'currency', 'payload_json']], left_on='bet_id', right_on='id')
    display(settled.tail(20))
    paper = settled.loc[settled['mode'].eq('paper')].copy()
    paper['pnl_amount'] = pd.to_numeric(paper['pnl_amount'])
    for currency, group in paper.groupby('currency'):
        group.sort_values('event_at')['pnl_amount'].cumsum().plot(title=f'Paper P&L — {currency}', ylabel=currency)
connection.close()